# Getting started

This notebook shows the basic `kpnn` workflow:

1. represent prior knowledge as an edgelist
2. compile the graph into a PyTorch model
3. use the compiled model like ordinary PyTorch
4. interpret the model back in biological space

The package philosophy is intentionally minimal:

- `kpnn` compiles biological structure into a PyTorch model
- PyTorch remains responsible for normal training workflows
- `kpnn` later maps attributions back to biological names


## Installation

During development, install the package in editable mode from the project root:

```bash
pip install -e .
```

For optional `AnnData` support:

```bash
pip install -e .[bio]
```


## Imports

In [1]:
import pandas as pd
import torch
from torch import nn
from IPython.display import display

from kpnn.compile_graph import compile_graph
from kpnn.interpret_model import interpret_model

/home/thomas/Documents/PhD/projects/KPNN/kpnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: define a prior-knowledge graph

For v1, `kpnn` expects an edgelist with two required columns:

- `source`
- `target`

Here we define a small feedforward biological graph with two genes, one
pathway node, and one output node.


In [2]:
edgelist = pd.DataFrame(
    {
        "source": ["gene_1", "gene_2", "pathway_1"],
        "target": ["pathway_1", "pathway_1", "output_1"],
    }
)

edgelist

,source,target
0,gene_1,pathway_1
1,gene_2,pathway_1
2,pathway_1,output_1


## Step 2: compile the graph

`compile_graph(...)` returns:

- a compiled PyTorch model
- a compilation artifact containing the metadata needed later for
  interpretation


In [3]:
model, artifact = compile_graph(
    edgelist=edgelist,
    backend="feedforward",
)

type(model), type(artifact)

[kpnn] Note: Graph contains 4 node(s) and 3 edge(s).
[kpnn] Note: Feedforward backend selected. Graph must be layerable.


(kpnn.nn.model.KPNNModel, kpnn.compile.artifact.KPNNArtifact)

## Inspect the artifact

The artifact is the bridge between compilation and interpretation.


In [4]:
artifact.backend

'feedforward'

In [5]:
artifact.feature_names

['gene_1', 'gene_2']

In [6]:
artifact.node_names_by_layer

{'layer_0': ['gene_1', 'gene_2'],
 'layer_1': ['pathway_1'],
 'layer_2': ['output_1']}

## Step 3: run the compiled model

The compiled model is a normal PyTorch module.


In [7]:
x = torch.randn(4, len(artifact.feature_names))
y = model(x)

y

tensor([[0.5009],
        [1.0464],
        [0.6714],
        [0.4150]], grad_fn=<AddmmBackward0>)

In [8]:
y.shape

torch.Size([4, 1])

## Step 4: use the compiled model inside ordinary PyTorch

`kpnn` does not impose activations, output heads, loss functions,
optimizers, or training loops.

A common pattern is to wrap the compiled model inside your own PyTorch
module.


In [9]:
class WrappedModel(nn.Module):
    def __init__(self, compiled_model):
        super().__init__()
        self.compiled_model = compiled_model
        self.activation = nn.ReLU()
        self.head = nn.Linear(1, 1)

    def forward(self, x):
        x = self.compiled_model(x)
        x = self.activation(x)
        x = self.head(x)
        return x


wrapped_model = WrappedModel(model)
wrapped_model

WrappedModel(
  (compiled_model): KPNNModel(
    (blocks): ModuleList(
      (0-1): 2 x FeedforwardLayerBlock(
        (linear): MaskedLinear()
      )
    )
  )
  (activation): ReLU()
  (head): Linear(in_features=1, out_features=1, bias=True)
)

### A minimal training step

This is only a small PyTorch example to show that the compiled model can be
used inside an ordinary training workflow.


In [10]:
optimizer = torch.optim.Adam(wrapped_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

x_train = torch.randn(8, len(artifact.feature_names))
y_true = torch.randn(8, 1)

optimizer.zero_grad()
y_pred = wrapped_model(x_train)
loss = loss_fn(y_pred, y_true)
loss.backward()
optimizer.step()

loss.item()

1.3063384294509888

## Step 5: feature-level interpretation

Feature-level interpretation maps attributions back to the input features.

Current intended combination:

- `target="features"`
- `method="integrated_gradients"`


In [11]:
feature_data = pd.DataFrame(
    {
        "gene_1": [0.1, 0.2, 0.3],
        "gene_2": [1.0, 1.1, 1.2],
    },
    index=["cell_1", "cell_2", "cell_3"],
)

feature_data

,gene_1,gene_2
cell_1,0.1,1.0
cell_2,0.2,1.1
cell_3,0.3,1.2


In [12]:
feature_attr = interpret_model(
    model=model,
    artifact=artifact,
    data=feature_data,
    target="features",
    method="integrated_gradients",
)

feature_attr

[kpnn] Finished interpretation with method 'integrated_gradients' for target 'features'.


,gene_1,gene_2
cell_1,0.007342,-0.304025
cell_2,0.014684,-0.334428
cell_3,0.022026,-0.364830


The returned object is a pandas DataFrame:

- rows = examples
- columns = feature names


## Step 6: node-level interpretation

Node-level interpretation maps attributions back to biological nodes inside
the compiled network.

Current intended combinations:

- `target="nodes", method="layer_conductance"`
- `target="nodes", method="layer_integrated_gradients"`


In [13]:
node_attr = interpret_model(
    model=model,
    artifact=artifact,
    data=feature_data,
    target="nodes",
    method="layer_conductance",
)

node_attr

[kpnn] Finished interpretation with method 'layer_conductance' for target 'nodes'.


{'layer_1':         pathway_1
 cell_1  -0.296360
 cell_2  -0.319395
 cell_3  -0.342431,
 'layer_2':         output_1
 cell_1 -0.296360
 cell_2 -0.319395
 cell_3 -0.342431}

The returned object is a dictionary:

- keys = layer names such as `"layer_1"` and `"layer_2"`
- values = pandas DataFrames with:
  - rows = examples
  - columns = biological node names in that layer

Internal pseudo nodes are filtered out of returned interpretation results.


In [14]:
for layer_name, layer_df in node_attr.items():
    print(layer_name)
    display(layer_df)

layer_1


,pathway_1
cell_1,-0.296360
cell_2,-0.319395
cell_3,-0.342431


layer_2


,output_1
cell_1,-0.296360
cell_2,-0.319395
cell_3,-0.342431


## Optional: layer integrated gradients

You can also compute node-level attributions with layer integrated
gradients.


In [15]:
node_attr_lig = interpret_model(
    model=model,
    artifact=artifact,
    data=feature_data,
    target="nodes",
    method="layer_integrated_gradients",
)

node_attr_lig

[kpnn] Finished interpretation with method 'layer_integrated_gradients' for target 'nodes'.


{'layer_1':         pathway_1
 cell_1  -0.296683
 cell_2  -0.319744
 cell_3  -0.342804,
 'layer_2':         output_1
 cell_1 -0.296683
 cell_2 -0.319744
 cell_3 -0.342804}

## Example with a skip edge

For feedforward compilation, skip edges are expanded internally into pseudo
nodes so the graph can still be computed layer by layer.

These pseudo nodes are internal only and do not appear in biological
interpretation outputs.


In [16]:
skip_edgelist = pd.DataFrame(
    {
        "source": [
            "gene_1",
            "pathway_1",
            "pathway_2",
            "gene_1",
        ],
        "target": [
            "pathway_1",
            "pathway_2",
            "output_1",
            "output_1",
        ],
    }
)

skip_model, skip_artifact = compile_graph(skip_edgelist)
skip_artifact.node_names_by_layer

[kpnn] Note: Graph contains 4 node(s) and 4 edge(s).
[kpnn] Note: Feedforward backend selected. Graph must be layerable.


{'layer_0': ['gene_1'],
 'layer_1': ['pathway_1', 'pseudo__gene_1__output_1__layer_1'],
 'layer_2': ['pathway_2', 'pseudo__gene_1__output_1__layer_2'],
 'layer_3': ['output_1']}

In [17]:
skip_data = pd.DataFrame(
    {"gene_1": [0.1, 0.2]},
    index=["cell_1", "cell_2"],
)

skip_node_attr = interpret_model(
    model=skip_model,
    artifact=skip_artifact,
    data=skip_data,
    target="nodes",
    method="layer_conductance",
)

skip_node_attr

[kpnn] Finished interpretation with method 'layer_conductance' for target 'nodes'.


{'layer_1':         pathway_1
 cell_1   0.002224
 cell_2   0.004448,
 'layer_2':         pathway_2
 cell_1   0.002224
 cell_2   0.004448,
 'layer_3':         output_1
 cell_1 -0.000130
 cell_2 -0.000261}

Notice that the returned node-level output contains only biological nodes,
not internal pseudo nodes.


## Summary

The basic `kpnn` workflow is:

1. define a biological edgelist
2. compile it into a PyTorch model with `compile_graph(...)`
3. train or wrap the model using ordinary PyTorch
4. map attributions back to biology with `interpret_model(...)`

This keeps the public API small while preserving normal PyTorch flexibility.